# Busca em Largura (BFS)
## Implementação em C e análise experimental com Python

Neste notebook, os conceitos centrais da **Busca em Largura (Breadth-First Search — BFS)** serão implementados em **linguagem C**.

Python será usado somente como ferramenta auxiliar para:

- ler arquivos CSV gerados pelos programas C;
- organizar resultados experimentais;
- visualizar camadas, distâncias e custos;
- comparar lista e matriz de adjacências.

### Objetivos

Ao final das atividades, você deverá ser capaz de:

- implementar uma **fila FIFO** sem usar bibliotecas prontas;
- representar grafos com **listas de adjacência**;
- implementar BFS e explicar por que ela visita o grafo por **camadas**;
- manter corretamente `visitado[]`, `dist[]` e `pai[]`;
- reconstruir caminhos usando o vetor de predecessores;
- explicar por que BFS encontra caminhos mínimos em **número de arestas** em grafos sem pesos;
- percorrer grafos desconexos construindo uma floresta de busca;
- explicar por que o vértice deve ser marcado como visitado **no momento do enfileiramento**;
- implementar BFS também com **matriz de adjacências**;
- comparar experimentalmente as complexidades:
  - lista: \(\Theta(|V|+|E|)\);
  - matriz: \(\Theta(|V|^2)\).

> **Regra:** fila, grafo, BFS e reconstrução dos caminhos devem ser implementados em C. Python não deve substituir esses algoritmos.

# 0. Executando C no Google Colab

O Colab utiliza Python por padrão, mas pode criar e executar programas C.

### Criar um arquivo

```text
%%writefile programa.c
```

grava o restante da célula no arquivo `programa.c`.

### Compilar

```text
!gcc -Wall -Wextra -O2 -std=c11 programa.c -o programa
```

- `-Wall -Wextra`: habilitam avisos úteis;
- `-O2`: ativa otimizações;
- `-std=c11`: utiliza o padrão C11;
- `-o programa`: nome do executável.

### Executar

```text
!./programa
```

O `!` permite executar um comando do sistema operacional dentro do notebook.

# Parte I — A fila FIFO

A BFS utiliza uma **fila FIFO** (*First In, First Out*).

O primeiro vértice descoberto é também o primeiro a ser removido para exploração. Essa propriedade é fundamental para que todos os vértices da camada \(k\) sejam processados antes dos vértices da camada \(k+1\).

Implementaremos uma fila circular usando um vetor.

Uma fila manterá:

- `dados`: vetor;
- `capacidade`: número máximo de elementos;
- `inicio`: posição do primeiro elemento;
- `fim`: posição em que ocorrerá a próxima inserção;
- `quantidade`: número de elementos atualmente armazenados.

In [ ]:
%%writefile fila.h
#ifndef FILA_H
#define FILA_H

typedef struct {
    int *dados;
    int capacidade;
    int inicio;
    int fim;
    int quantidade;
} Fila;

Fila *fila_cria(int capacidade);
int fila_vazia(const Fila *f);
int fila_cheia(const Fila *f);
int fila_enfileira(Fila *f, int valor);
int fila_desenfileira(Fila *f, int *valor);
void fila_imprime(const Fila *f);
void fila_libera(Fila *f);

#endif

# Exercício 1 — Implemente a fila

Complete `fila.c`.

### Requisitos

- não use estruturas prontas;
- use alocação dinâmica;
- trate fila vazia e fila cheia;
- a fila deve ser **circular**.

A atualização circular pode ser feita com:

```c
(indice + 1) % capacidade
```

### Antes de programar

Para uma fila de capacidade 5, simule no papel:

1. enfileira A, B, C;
2. remove A e B;
3. enfileira D, E, F.

Observe que o vetor pode reutilizar posições do início.

In [ ]:
%%writefile fila.c
#include <stdio.h>
#include <stdlib.h>
#include "fila.h"

Fila *fila_cria(int capacidade) {
    /* TODO */
    return NULL;
}

int fila_vazia(const Fila *f) {
    /* TODO */
    return 1;
}

int fila_cheia(const Fila *f) {
    /* TODO */
    return 0;
}

int fila_enfileira(Fila *f, int valor) {
    /* TODO:
       Retorne 1 em caso de sucesso e 0 se a fila estiver cheia.
    */
    return 0;
}

int fila_desenfileira(Fila *f, int *valor) {
    /* TODO:
       Retorne 1 em caso de sucesso e 0 se a fila estiver vazia.
    */
    return 0;
}

void fila_imprime(const Fila *f) {
    /* TODO:
       Imprima na ordem em que os elementos serão removidos.
    */
}

void fila_libera(Fila *f) {
    /* TODO */
}

In [ ]:
%%writefile teste_fila.c
#include <stdio.h>
#include "fila.h"

int main(void) {
    Fila *f = fila_cria(5);
    int x;

    if (!f) return 1;

    fila_enfileira(f, 10);
    fila_enfileira(f, 20);
    fila_enfileira(f, 30);

    printf("Fila inicial: ");
    fila_imprime(f);

    fila_desenfileira(f, &x);
    printf("Removeu %d\n", x);

    fila_desenfileira(f, &x);
    printf("Removeu %d\n", x);

    fila_enfileira(f, 40);
    fila_enfileira(f, 50);
    fila_enfileira(f, 60);

    printf("Após reutilizar posições: ");
    fila_imprime(f);

    printf("Vazia? %s\n", fila_vazia(f) ? "sim" : "não");
    printf("Cheia? %s\n", fila_cheia(f) ? "sim" : "não");

    fila_libera(f);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 fila.c teste_fila.c -o teste_fila
!./teste_fila

# Parte II — Representação por lista de adjacências

Usaremos uma lista ligada para representar os vizinhos de cada vértice.

```c
typedef struct NoAdj {
    int vertice;
    struct NoAdj *prox;
} NoAdj;
```

O grafo mantém, para cada vértice, o início e o fim da lista de adjacências.

Manter também o `fim` permite inserir vizinhos na ordem em que as arestas são adicionadas. Isso é útil porque a ordem dos vizinhos pode modificar a **ordem de visita e a árvore de predecessores**, embora não modifique as distâncias mínimas.

Usaremos os vértices:

```text
A=0, B=1, C=2, D=3, E=4, F=5, G=6, H=7
```

In [ ]:
%%writefile grafo_lista.h
#ifndef GRAFO_LISTA_H
#define GRAFO_LISTA_H

typedef struct NoAdj {
    int vertice;
    struct NoAdj *prox;
} NoAdj;

typedef struct {
    int n;
    NoAdj **inicio;
    NoAdj **fim;
} GrafoLista;

GrafoLista *grafo_lista_cria(int n);
int grafo_lista_adiciona_aresta(GrafoLista *g, int u, int v);
void grafo_lista_imprime(const GrafoLista *g);
void grafo_lista_libera(GrafoLista *g);

#endif

# Exercício 2 — Construa o grafo com listas ligadas

Implemente:

- criação;
- inserção de uma aresta **não dirigida**;
- impressão;
- liberação de memória.

Para uma aresta `u-v`, é necessário inserir:

```text
v na lista de u
u na lista de v
```

No exemplo usado nas atividades, as arestas serão adicionadas nesta ordem:

```text
A-B
A-C
B-D
B-E
C-E
C-F
D-G
E-G
E-H
F-H
```

Com inserção no final, obtemos as listas:

```text
A: B, C
B: A, D, E
C: A, E, F
D: B, G
E: B, C, G, H
F: C, H
G: D, E
H: E, F
```

In [ ]:
%%writefile grafo_lista.c
#include <stdio.h>
#include <stdlib.h>
#include "grafo_lista.h"

static int adiciona_direcionada(GrafoLista *g, int u, int v) {
    /* TODO */
    return 0;
}

GrafoLista *grafo_lista_cria(int n) {
    /* TODO */
    return NULL;
}

int grafo_lista_adiciona_aresta(GrafoLista *g, int u, int v) {
    /* TODO:
       Grafo não dirigido: adicione u->v e v->u.
    */
    return 0;
}

void grafo_lista_imprime(const GrafoLista *g) {
    /* TODO */
}

void grafo_lista_libera(GrafoLista *g) {
    /* TODO */
}

In [ ]:
%%writefile teste_grafo.c
#include <stdio.h>
#include "grafo_lista.h"

static char nome(int v) {
    return (char)('A' + v);
}

int main(void) {
    GrafoLista *g = grafo_lista_cria(8);
    if (!g) return 1;

    int arestas[][2] = {
        {0,1}, {0,2}, {1,3}, {1,4}, {2,4},
        {2,5}, {3,6}, {4,6}, {4,7}, {5,7}
    };

    int m = sizeof(arestas) / sizeof(arestas[0]);

    for (int i = 0; i < m; i++)
        grafo_lista_adiciona_aresta(g, arestas[i][0], arestas[i][1]);

    printf("Listas de adjacência:\n");

    for (int u = 0; u < g->n; u++) {
        printf("%c: ", nome(u));

        for (NoAdj *p = g->inicio[u]; p; p = p->prox)
            printf("%c ", nome(p->vertice));

        printf("\n");
    }

    grafo_lista_libera(g);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 grafo_lista.c teste_grafo.c -o teste_grafo
!./teste_grafo

# Parte III — Estado mantido pela BFS

Durante a BFS, manteremos:

### `visitado[v]`

Indica se `v` já foi **descoberto e colocado na fila**.

### `dist[v]`

Número mínimo de arestas conhecido da origem até `v`.

Inicialização:

```text
dist[origem] = 0
dist[demais] = -1
```

Usaremos `-1` para representar distância ainda desconhecida.

### `pai[v]`

Vértice que descobriu `v`.

```text
pai[origem] = -1
```

O vetor `pai[]` permitirá reconstruir o caminho.

### `ordem[]`

Guardará a ordem em que os vértices são retirados da fila.

> **Importante:** marque um vértice como visitado no momento em que ele é **enfileirado**, não quando for removido.

In [ ]:
%%writefile bfs.h
#ifndef BFS_H
#define BFS_H

#include "grafo_lista.h"

void bfs_lista(
    const GrafoLista *g,
    int origem,
    int visitado[],
    int dist[],
    int pai[],
    int ordem[],
    int *tam_ordem
);

#endif

# Exercício 3 — Implemente a BFS

Use a fila desenvolvida no Exercício 1.

Pseudocódigo:

```text
para cada v:
    visitado[v] = falso
    dist[v] = infinito
    pai[v] = nulo

visitado[s] = verdadeiro
dist[s] = 0
enfileirar(s)

enquanto fila não estiver vazia:
    u = desenfileirar()

    para cada v em adj[u]:
        se v não foi visitado:
            visitado[v] = verdadeiro
            dist[v] = dist[u] + 1
            pai[v] = u
            enfileirar(v)
```

### Questão antes da implementação

Por que `visitado[v] = verdadeiro` deve aparecer **antes** de `enfileirar(v)`?

In [ ]:
%%writefile bfs.c
#include <stdlib.h>
#include "fila.h"
#include "bfs.h"

void bfs_lista(
    const GrafoLista *g,
    int origem,
    int visitado[],
    int dist[],
    int pai[],
    int ordem[],
    int *tam_ordem
) {
    /* TODO */
}

In [ ]:
%%writefile teste_bfs.c
#include <stdio.h>
#include "grafo_lista.h"
#include "bfs.h"

static char nome(int v) {
    return (char)('A' + v);
}

int main(void) {
    GrafoLista *g = grafo_lista_cria(8);

    int arestas[][2] = {
        {0,1}, {0,2}, {1,3}, {1,4}, {2,4},
        {2,5}, {3,6}, {4,6}, {4,7}, {5,7}
    };

    int m = sizeof(arestas) / sizeof(arestas[0]);

    for (int i = 0; i < m; i++)
        grafo_lista_adiciona_aresta(g, arestas[i][0], arestas[i][1]);

    int visitado[8], dist[8], pai[8], ordem[8], tam = 0;

    bfs_lista(g, 0, visitado, dist, pai, ordem, &tam);

    printf("Ordem de remoção da fila:\n");
    for (int i = 0; i < tam; i++)
        printf("%c ", nome(ordem[i]));

    printf("\n\nVértice  dist  pai\n");

    for (int v = 0; v < 8; v++) {
        printf("   %c      %2d    ", nome(v), dist[v]);

        if (pai[v] == -1)
            printf("-\n");
        else
            printf("%c\n", nome(pai[v]));
    }

    grafo_lista_libera(g);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    fila.c grafo_lista.c bfs.c teste_bfs.c \
    -o teste_bfs

!./teste_bfs

## Resultado esperado para o exemplo

Se as listas de adjacência estiverem na ordem especificada, a BFS iniciada em `A` deve produzir:

```text
ordem: A B C D E F G H
```

Distâncias:

```text
A:0
B:1
C:1
D:2
E:2
F:2
G:3
H:3
```

O importante não é decorar a ordem. Observe a propriedade:

> todos os vértices com distância \(k\) são descobertos antes dos vértices com distância \(k+1\).

# Exercício 4 — Exporte o resultado da BFS para CSV

A implementação continua sendo C. O programa abaixo chama sua BFS e grava:

```text
vertice,distancia,pai,ordem
```

em `resultado_bfs.csv`.

O Python será usado depois apenas para visualizar as camadas.

In [ ]:
%%writefile exporta_bfs.c
#include <stdio.h>
#include "grafo_lista.h"
#include "bfs.h"

int main(void) {
    GrafoLista *g = grafo_lista_cria(8);

    int arestas[][2] = {
        {0,1}, {0,2}, {1,3}, {1,4}, {2,4},
        {2,5}, {3,6}, {4,6}, {4,7}, {5,7}
    };

    int m = sizeof(arestas) / sizeof(arestas[0]);

    for (int i = 0; i < m; i++)
        grafo_lista_adiciona_aresta(g, arestas[i][0], arestas[i][1]);

    int visitado[8], dist[8], pai[8], ordem[8], tam = 0;

    bfs_lista(g, 0, visitado, dist, pai, ordem, &tam);

    int pos_ordem[8];

    for (int i = 0; i < tam; i++)
        pos_ordem[ordem[i]] = i;

    FILE *arq = fopen("resultado_bfs.csv", "w");
    if (!arq) return 1;

    fprintf(arq, "vertice,distancia,pai,ordem\n");

    for (int v = 0; v < 8; v++) {
        char pai_nome = pai[v] == -1 ? '-' : (char)('A' + pai[v]);

        fprintf(
            arq, "%c,%d,%c,%d\n",
            'A' + v, dist[v], pai_nome, pos_ordem[v]
        );
    }

    fclose(arq);
    grafo_lista_libera(g);

    printf("Arquivo resultado_bfs.csv criado.\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    fila.c grafo_lista.c bfs.c exporta_bfs.c \
    -o exporta_bfs

!./exporta_bfs

# Python — lendo o CSV com `pandas`

`pandas` é uma biblioteca para manipulação de dados tabulares.

```python
import pandas as pd
```

O apelido `pd` é uma convenção.

```python
df = pd.read_csv("resultado_bfs.csv")
```

cria um **DataFrame**, uma estrutura semelhante a uma tabela com linhas e colunas.

O método `sort_values()` permite ordenar a visualização por uma coluna. Aqui ordenaremos pela ordem de remoção da fila.

In [ ]:
import pandas as pd

bfs_df = pd.read_csv("resultado_bfs.csv")

display(
    bfs_df.sort_values("ordem")
)

# Python — visualizando as camadas com `matplotlib`

Não usaremos Python para calcular a BFS. As distâncias já foram calculadas em C.

`matplotlib` apenas transformará esses resultados em uma figura.

Usaremos:

- `plt.scatter(x,y)`: desenha pontos;
- `plt.annotate(texto, ...)`: escreve o nome do vértice próximo ao ponto;
- `plt.xlabel()` e `plt.ylabel()`: nomes dos eixos;
- `plt.show()`: exibe a figura.

No eixo x colocaremos a distância. Assim, vértices na mesma coluna pertencem à mesma camada.

In [ ]:
import matplotlib.pyplot as plt

camadas = bfs_df.sort_values(["distancia", "ordem"]).copy()

camadas["pos_na_camada"] = (
    camadas.groupby("distancia").cumcount()
)

plt.figure(figsize=(9, 5))

plt.scatter(
    camadas["distancia"],
    camadas["pos_na_camada"],
    s=350
)

for _, linha in camadas.iterrows():
    plt.annotate(
        linha["vertice"],
        (linha["distancia"], linha["pos_na_camada"]),
        ha="center",
        va="center"
    )

plt.xlabel("Distância mínima em número de arestas")
plt.ylabel("Posição apenas para visualização")
plt.title("Camadas produzidas pela BFS a partir de A")
plt.xticks(sorted(camadas["distancia"].unique()))
plt.grid(axis="x")
plt.show()

# Parte IV — Reconstrução de caminhos

O vetor `pai[]` descreve uma árvore de predecessores.

Para reconstruir um caminho até um vértice destino:

```text
destino
pai[destino]
pai[pai[destino]]
...
origem
```

Essa sequência aparece na ordem inversa. Podemos armazená-la em um vetor e depois imprimir de trás para frente.

No exemplo:

```text
pai[H] = E
pai[E] = B
pai[B] = A
```

Logo:

```text
A -> B -> E -> H
```

# Exercício 5 — Reconstrua o caminho

Implemente uma função que:

- receba `pai[]`, origem e destino;
- retorne o tamanho do caminho;
- escreva os vértices do caminho em um vetor;
- retorne 0 se o destino não for alcançável.

Não execute outra BFS dentro da função. Use apenas o vetor `pai[]`.

In [ ]:
%%writefile caminho.c
#include <stdio.h>

int reconstroi_caminho(
    const int pai[],
    int origem,
    int destino,
    int caminho[],
    int capacidade
) {
    /* TODO */
    return 0;
}

In [ ]:
%%writefile teste_caminho.c
#include <stdio.h>
#include "grafo_lista.h"
#include "bfs.h"

int reconstroi_caminho(
    const int pai[],
    int origem,
    int destino,
    int caminho[],
    int capacidade
);

int main(void) {
    GrafoLista *g = grafo_lista_cria(8);

    int arestas[][2] = {
        {0,1}, {0,2}, {1,3}, {1,4}, {2,4},
        {2,5}, {3,6}, {4,6}, {4,7}, {5,7}
    };

    int m = sizeof(arestas) / sizeof(arestas[0]);

    for (int i = 0; i < m; i++)
        grafo_lista_adiciona_aresta(g, arestas[i][0], arestas[i][1]);

    int vis[8], dist[8], pai[8], ordem[8], tam = 0;

    bfs_lista(g, 0, vis, dist, pai, ordem, &tam);

    int caminho[8];
    int n = reconstroi_caminho(pai, 0, 7, caminho, 8);

    printf("Caminho A -> H: ");

    for (int i = 0; i < n; i++) {
        if (i) printf(" -> ");
        printf("%c", 'A' + caminho[i]);
    }

    printf("\nNúmero de arestas: %d\n", n > 0 ? n - 1 : -1);

    grafo_lista_libera(g);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    fila.c grafo_lista.c bfs.c caminho.c teste_caminho.c \
    -o teste_caminho

!./teste_caminho

# Parte V — Por que marcar no enfileiramento?

Considere o grafo:

```text
    B
   / \
  A   D
   \ /
    C
```

A origem é `A`.

Se `D` só for marcado quando sair da fila:

1. `B` descobre `D` e o enfileira;
2. antes de `D` sair da fila, `C` também encontra `D` ainda "não visitado";
3. `C` enfileira `D` novamente.

A BFS ainda pode terminar, mas realiza trabalho desnecessário e a fila pode conter duplicatas.

Execute o experimento abaixo e compare.

In [ ]:
%%writefile marcacao_tardia.c
#include <stdio.h>
#include <stdlib.h>
#include "fila.h"
#include "grafo_lista.h"

static int bfs_correta(const GrafoLista *g, int s) {
    int *vis = calloc(g->n, sizeof(int));
    Fila *f = fila_cria(g->n * 4);
    int enfileiramentos = 0;

    vis[s] = 1;
    fila_enfileira(f, s);
    enfileiramentos++;

    while (!fila_vazia(f)) {
        int u;
        fila_desenfileira(f, &u);

        for (NoAdj *p = g->inicio[u]; p; p = p->prox) {
            int v = p->vertice;

            if (!vis[v]) {
                vis[v] = 1;  /* marca ANTES */
                fila_enfileira(f, v);
                enfileiramentos++;
            }
        }
    }

    free(vis);
    fila_libera(f);
    return enfileiramentos;
}

static int bfs_marcando_tarde(const GrafoLista *g, int s) {
    int *vis = calloc(g->n, sizeof(int));
    Fila *f = fila_cria(g->n * 10);
    int enfileiramentos = 0;

    fila_enfileira(f, s);
    enfileiramentos++;

    while (!fila_vazia(f)) {
        int u;
        fila_desenfileira(f, &u);

        if (vis[u])
            continue;

        vis[u] = 1;  /* ERRADO: marca somente ao remover */

        for (NoAdj *p = g->inicio[u]; p; p = p->prox) {
            int v = p->vertice;

            if (!vis[v]) {
                fila_enfileira(f, v);
                enfileiramentos++;
            }
        }
    }

    free(vis);
    fila_libera(f);
    return enfileiramentos;
}

int main(void) {
    GrafoLista *g = grafo_lista_cria(4);

    grafo_lista_adiciona_aresta(g, 0, 1); /* A-B */
    grafo_lista_adiciona_aresta(g, 0, 2); /* A-C */
    grafo_lista_adiciona_aresta(g, 1, 3); /* B-D */
    grafo_lista_adiciona_aresta(g, 2, 3); /* C-D */

    printf("Marcação no enfileiramento: %d inserções na fila\n",
           bfs_correta(g, 0));

    printf("Marcação tardia:           %d inserções na fila\n",
           bfs_marcando_tarde(g, 0));

    grafo_lista_libera(g);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    fila.c grafo_lista.c marcacao_tardia.c \
    -o marcacao_tardia

!./marcacao_tardia

## Questões sobre a marcação

1. Por que `D` pode entrar mais de uma vez na fila?
2. O problema ficaria mais grave em grafos com muitos caminhos convergindo para os mesmos vértices?
3. Qual é a regra segura?

> **Regra:** se `v` será colocado na fila, marque `visitado[v]` **antes** de inseri-lo.

# Parte VI — Grafos desconexos

Uma BFS iniciada em `s` alcança apenas a **componente conexa** que contém `s`.

Para percorrer todo o grafo:

```text
para cada vértice v:
    se v ainda não foi visitado:
        execute uma BFS a partir de v
```

O resultado é uma **floresta de busca**.

Além de visitar todo o grafo, esse procedimento permite numerar as componentes conexas.

# Exercício 6 — Componentes conexas com BFS

Implemente:

```c
int componentes_bfs(const GrafoLista *g, int componente[]);
```

A função deve:

- percorrer todo o grafo;
- atribuir um identificador de componente a cada vértice;
- retornar o número total de componentes.

Use sua própria fila.

In [ ]:
%%writefile componentes.c
#include <stdlib.h>
#include "fila.h"
#include "grafo_lista.h"

int componentes_bfs(
    const GrafoLista *g,
    int componente[]
) {
    /* TODO */
    return 0;
}

In [ ]:
%%writefile teste_componentes.c
#include <stdio.h>
#include "grafo_lista.h"

int componentes_bfs(
    const GrafoLista *g,
    int componente[]
);

int main(void) {
    GrafoLista *g = grafo_lista_cria(7);

    /* componente 0: A-B-C-D */
    grafo_lista_adiciona_aresta(g, 0, 1);
    grafo_lista_adiciona_aresta(g, 1, 2);
    grafo_lista_adiciona_aresta(g, 2, 3);
    grafo_lista_adiciona_aresta(g, 3, 0);

    /* componente 1: E-F-G */
    grafo_lista_adiciona_aresta(g, 4, 5);
    grafo_lista_adiciona_aresta(g, 5, 6);

    int componente[7];

    int qtd = componentes_bfs(g, componente);

    printf("Número de componentes: %d\n", qtd);

    for (int v = 0; v < 7; v++)
        printf("%c -> componente %d\n", 'A' + v, componente[v]);

    grafo_lista_libera(g);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    fila.c grafo_lista.c componentes.c teste_componentes.c \
    -o teste_componentes

!./teste_componentes

# Parte VII — Matriz de adjacências

Em uma matriz de adjacências:

```text
matriz[u][v] = 1  se existe aresta u-v
matriz[u][v] = 0  caso contrário
```

Para um grafo não dirigido, a matriz é simétrica.

Uma representação linear contígua pode ser acessada por:

```c
matriz[u * n + v]
```

### Consequência para a BFS

Ao remover `u` da fila, precisamos testar:

```text
v = 0, 1, ..., n-1
```

para descobrir seus vizinhos.

Mesmo que `u` tenha poucos vizinhos, examinamos toda a linha. Por isso, a BFS com matriz tem custo \(\Theta(|V|^2)\).

In [ ]:
%%writefile grafo_matriz.h
#ifndef GRAFO_MATRIZ_H
#define GRAFO_MATRIZ_H

typedef struct {
    int n;
    unsigned char *matriz;
} GrafoMatriz;

GrafoMatriz *grafo_matriz_cria(int n);
void grafo_matriz_adiciona_aresta(GrafoMatriz *g, int u, int v);
int grafo_matriz_tem_aresta(const GrafoMatriz *g, int u, int v);
void grafo_matriz_libera(GrafoMatriz *g);

#endif

# Exercício 7 — Matriz + BFS

Implemente:

1. criação da matriz;
2. inserção de aresta não dirigida;
3. consulta de aresta;
4. liberação;
5. BFS usando matriz.

Use os mesmos vetores:

```text
visitado[]
dist[]
pai[]
ordem[]
```

e verifique se as distâncias são iguais às obtidas com a lista.

In [ ]:
%%writefile grafo_matriz.c
#include <stdlib.h>
#include "grafo_matriz.h"

GrafoMatriz *grafo_matriz_cria(int n) {
    /* TODO */
    return NULL;
}

void grafo_matriz_adiciona_aresta(GrafoMatriz *g, int u, int v) {
    /* TODO */
}

int grafo_matriz_tem_aresta(const GrafoMatriz *g, int u, int v) {
    /* TODO */
    return 0;
}

void grafo_matriz_libera(GrafoMatriz *g) {
    /* TODO */
}

In [ ]:
%%writefile bfs_matriz.c
#include <stdlib.h>
#include "fila.h"
#include "grafo_matriz.h"

void bfs_matriz(
    const GrafoMatriz *g,
    int origem,
    int visitado[],
    int dist[],
    int pai[],
    int ordem[],
    int *tam_ordem
) {
    /* TODO */
}

In [ ]:
%%writefile teste_bfs_matriz.c
#include <stdio.h>
#include "grafo_matriz.h"

void bfs_matriz(
    const GrafoMatriz *g,
    int origem,
    int visitado[],
    int dist[],
    int pai[],
    int ordem[],
    int *tam_ordem
);

int main(void) {
    GrafoMatriz *g = grafo_matriz_cria(8);

    int arestas[][2] = {
        {0,1}, {0,2}, {1,3}, {1,4}, {2,4},
        {2,5}, {3,6}, {4,6}, {4,7}, {5,7}
    };

    int m = sizeof(arestas) / sizeof(arestas[0]);

    for (int i = 0; i < m; i++)
        grafo_matriz_adiciona_aresta(g, arestas[i][0], arestas[i][1]);

    int vis[8], dist[8], pai[8], ordem[8], tam = 0;

    bfs_matriz(g, 0, vis, dist, pai, ordem, &tam);

    printf("Distâncias a partir de A:\n");

    for (int v = 0; v < 8; v++)
        printf("%c: %d\n", 'A' + v, dist[v]);

    grafo_matriz_libera(g);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    fila.c grafo_matriz.c bfs_matriz.c teste_bfs_matriz.c \
    -o teste_bfs_matriz

!./teste_bfs_matriz

# Exercício 8 — Experimento: lista × matriz

Agora faremos uma comparação experimental.

O programa abaixo cria **o mesmo grafo esparso** nas duas representações.

Para cada tamanho:

- garante conectividade adicionando uma cadeia;
- adiciona arestas aleatórias até aproximadamente \(2|V|\) arestas;
- executa BFS;
- mede:
  - tempo;
  - número de verificações de vizinhança.

### Métrica estrutural

Na lista, uma verificação corresponde a examinar uma entrada de lista de adjacência.

Na matriz, uma verificação corresponde a testar `matriz[u][v]`.

Essa métrica ajuda a observar a diferença de complexidade mesmo quando o relógio possui ruído.

In [ ]:
%%writefile benchmark_bfs.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include "fila.h"
#include "grafo_lista.h"
#include "grafo_matriz.h"

static long long bfs_lista_conta(const GrafoLista *g, int s) {
    int *vis = calloc(g->n, sizeof(int));
    Fila *f = fila_cria(g->n);
    long long checks = 0;

    vis[s] = 1;
    fila_enfileira(f, s);

    while (!fila_vazia(f)) {
        int u;
        fila_desenfileira(f, &u);

        for (NoAdj *p = g->inicio[u]; p; p = p->prox) {
            checks++;
            int v = p->vertice;

            if (!vis[v]) {
                vis[v] = 1;
                fila_enfileira(f, v);
            }
        }
    }

    free(vis);
    fila_libera(f);
    return checks;
}

static long long bfs_matriz_conta(const GrafoMatriz *g, int s) {
    int *vis = calloc(g->n, sizeof(int));
    Fila *f = fila_cria(g->n);
    long long checks = 0;

    vis[s] = 1;
    fila_enfileira(f, s);

    while (!fila_vazia(f)) {
        int u;
        fila_desenfileira(f, &u);

        for (int v = 0; v < g->n; v++) {
            checks++;

            if (grafo_matriz_tem_aresta(g, u, v) && !vis[v]) {
                vis[v] = 1;
                fila_enfileira(f, v);
            }
        }
    }

    free(vis);
    fila_libera(f);
    return checks;
}

static double us(clock_t a, clock_t b) {
    return 1e6 * (double)(b - a) / CLOCKS_PER_SEC;
}

int main(void) {
    const int tamanhos[] = {100, 250, 500, 750, 1000};
    const int nt = sizeof(tamanhos) / sizeof(tamanhos[0]);
    const int repeticoes = 5;

    FILE *arq = fopen("benchmark_bfs.csv", "w");
    if (!arq) return 1;

    fprintf(arq, "representacao,V,E,repeticao,tempo_us,verificacoes\n");

    srand(42);

    for (int k = 0; k < nt; k++) {
        int n = tamanhos[k];
        int alvo_arestas = 2 * n;

        for (int rep = 1; rep <= repeticoes; rep++) {
            GrafoLista *gl = grafo_lista_cria(n);
            GrafoMatriz *gm = grafo_matriz_cria(n);

            unsigned char *usada = calloc((size_t)n * n, 1);
            int e = 0;

            /* cadeia para garantir conectividade */
            for (int i = 0; i < n - 1; i++) {
                grafo_lista_adiciona_aresta(gl, i, i + 1);
                grafo_matriz_adiciona_aresta(gm, i, i + 1);
                usada[(size_t)i * n + (i + 1)] = 1;
                usada[(size_t)(i + 1) * n + i] = 1;
                e++;
            }

            while (e < alvo_arestas) {
                int u = rand() % n;
                int v = rand() % n;

                if (u == v) continue;

                if (!usada[(size_t)u * n + v]) {
                    usada[(size_t)u * n + v] = 1;
                    usada[(size_t)v * n + u] = 1;

                    grafo_lista_adiciona_aresta(gl, u, v);
                    grafo_matriz_adiciona_aresta(gm, u, v);
                    e++;
                }
            }

            clock_t a = clock();
            long long c_lista = bfs_lista_conta(gl, 0);
            clock_t b = clock();

            fprintf(arq, "lista,%d,%d,%d,%.3f,%lld\n",
                    n, e, rep, us(a,b), c_lista);

            a = clock();
            long long c_matriz = bfs_matriz_conta(gm, 0);
            b = clock();

            fprintf(arq, "matriz,%d,%d,%d,%.3f,%lld\n",
                    n, e, rep, us(a,b), c_matriz);

            free(usada);
            grafo_lista_libera(gl);
            grafo_matriz_libera(gm);
        }
    }

    fclose(arq);
    printf("Arquivo benchmark_bfs.csv criado.\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    fila.c grafo_lista.c grafo_matriz.c benchmark_bfs.c \
    -o benchmark_bfs

!./benchmark_bfs

# Python — resumindo repetições com `groupby`

Cada configuração foi executada várias vezes.

O método:

```python
df.groupby(["representacao", "V"])
```

agrupa linhas que possuem a mesma representação e o mesmo tamanho.

Depois:

```python
.agg(...)
```

calcula estatísticas em cada grupo.

Usaremos a média do tempo e das verificações.

In [ ]:
benchmark = pd.read_csv("benchmark_bfs.csv")

resumo = (
    benchmark.groupby(["representacao", "V"])
             .agg(
                 tempo_medio_us=("tempo_us", "mean"),
                 verificacoes_medias=("verificacoes", "mean"),
                 E=("E", "mean"),
             )
             .reset_index()
)

display(resumo)

# Python — gráfico de verificações

O número de verificações é particularmente didático:

- na lista: aproximadamente \(2|E|\) em um grafo não dirigido;
- na matriz: aproximadamente \(|V|^2\).

Usaremos um gráfico de linhas.

In [ ]:
plt.figure(figsize=(8, 5))

for representacao in resumo["representacao"].unique():
    parte = resumo[resumo["representacao"] == representacao]

    plt.plot(
        parte["V"],
        parte["verificacoes_medias"],
        marker="o",
        label=representacao
    )

plt.xlabel("Número de vértices |V|")
plt.ylabel("Verificações de vizinhança")
plt.title("BFS: lista × matriz de adjacências")
plt.legend()
plt.grid()
plt.show()

# Python — gráfico de tempo

O tempo pode apresentar ruído e depende da máquina. Ele deve ser interpretado juntamente com a contagem de operações.

In [ ]:
plt.figure(figsize=(8, 5))

for representacao in resumo["representacao"].unique():
    parte = resumo[resumo["representacao"] == representacao]

    plt.plot(
        parte["V"],
        parte["tempo_medio_us"],
        marker="o",
        label=representacao
    )

plt.xlabel("Número de vértices |V|")
plt.ylabel("Tempo médio (µs)")
plt.title("Tempo experimental da BFS")
plt.legend()
plt.grid()
plt.show()

# Exercício 9 — Detecção de ciclo em grafo não dirigido

Durante uma BFS em um grafo **não dirigido**, se ao examinar a aresta `u-v`:

- `v` já foi visitado;
- e `v != pai[u]`;

então encontramos evidência de um ciclo.

Implemente:

```c
int possui_ciclo_bfs(const GrafoLista *g);
```

A função deve funcionar também em grafos desconexos, portanto precisa iniciar uma BFS em cada componente ainda não visitada.

In [ ]:
%%writefile ciclo_bfs.c
#include <stdlib.h>
#include "fila.h"
#include "grafo_lista.h"

int possui_ciclo_bfs(const GrafoLista *g) {
    /* TODO */
    return 0;
}

In [ ]:
%%writefile teste_ciclo.c
#include <stdio.h>
#include "grafo_lista.h"

int possui_ciclo_bfs(const GrafoLista *g);

int main(void) {
    GrafoLista *arvore = grafo_lista_cria(5);
    grafo_lista_adiciona_aresta(arvore, 0, 1);
    grafo_lista_adiciona_aresta(arvore, 1, 2);
    grafo_lista_adiciona_aresta(arvore, 1, 3);
    grafo_lista_adiciona_aresta(arvore, 3, 4);

    GrafoLista *ciclo = grafo_lista_cria(4);
    grafo_lista_adiciona_aresta(ciclo, 0, 1);
    grafo_lista_adiciona_aresta(ciclo, 1, 2);
    grafo_lista_adiciona_aresta(ciclo, 2, 0);
    grafo_lista_adiciona_aresta(ciclo, 2, 3);

    printf("Árvore possui ciclo? %s\n",
           possui_ciclo_bfs(arvore) ? "sim" : "não");

    printf("Segundo grafo possui ciclo? %s\n",
           possui_ciclo_bfs(ciclo) ? "sim" : "não");

    grafo_lista_libera(arvore);
    grafo_lista_libera(ciclo);

    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    fila.c grafo_lista.c ciclo_bfs.c teste_ciclo.c \
    -o teste_ciclo

!./teste_ciclo

# Exercício 10 — BFS minimiza arestas, não soma de pesos

Considere:

```text
A --100-- B
 \       /
  1     1
   \   /
     C
```

Há dois caminhos de `A` até `B`:

```text
A -> B       : 1 aresta, custo 100
A -> C -> B  : 2 arestas, custo 2
```

A BFS prefere o primeiro porque minimiza **número de arestas**.

### Responda

1. A BFS está errada ao devolver `A -> B`?
2. Que quantidade ela realmente minimiza?
3. Por que não podemos usar BFS diretamente quando pesos diferentes representam custos?
4. Pesquise posteriormente qual algoritmo é apropriado quando os pesos são não negativos.

> Não implemente outro algoritmo nesta atividade. O objetivo é compreender o limite de aplicação da BFS.

# Questões finais

1. Qual é o papel da fila FIFO na descoberta por camadas?
2. Por que marcar no enfileiramento evita duplicatas?
3. O que `dist[v]` representa?
4. O que `pai[v]` permite fazer?
5. Por que a ordem das listas de adjacência pode mudar a árvore BFS?
6. Essa ordem pode mudar a distância mínima encontrada?
7. O que ocorre se o grafo for desconexo?
8. Por que BFS encontra caminhos mínimos em número de arestas?
9. Compare a quantidade de verificações de vizinhança entre lista e matriz.
10. Quando uma matriz de adjacências ainda pode ser uma escolha interessante?
11. Qual é o espaço adicional da BFS além da própria representação do grafo?
12. Qual a diferença entre:
    - "vértice descoberto";
    - "vértice retirado da fila";
    - "vértice ainda não alcançado"?

# Desafio opcional — graus de separação

Crie em C um pequeno grafo representando pessoas conectadas por amizade.

Use BFS para responder:

- quantos graus de separação existem entre duas pessoas?
- qual sequência de pessoas forma um caminho mínimo?
- quais pessoas estão exatamente a distância 2 de uma determinada origem?

Depois exporte:

```text
pessoa,distancia
```

para CSV e faça **apenas a visualização** das distâncias em Python.

Esse exercício aproxima BFS de aplicações de redes sociais e Sistemas de Informação.